<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/05%20-%20Formas%20Normais%20e%20Otimizacao%20Booleana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 05 - Notebook: Otimização Booleana e Formas Normais (SymPy)

Este notebook tem como objetivo aplicar a álgebra booleana de forma algorítmica para otimizar as regras de controle do SCADA-Core do AGV. Expressões otimizadas reduzem a carga de processamento do CLP embarcado e minimizam a latência de varredura.

Utilizaremos a biblioteca `sympy` para processamento simbólico.

In [1]:
# Instalação e importação da biblioteca simbólica (SymPy)
!pip install sympy -q

import sympy as sp
from sympy.logic.boolalg import to_dnf, to_cnf, simplify_logic

print("Biblioteca SymPy carregada com sucesso!")

Biblioteca SymPy carregada com sucesso!


## 1. Simplificação Algorítmica da Lógica de Tração

Vamos inserir a equação não otimizada do comando do motor ($cmd_{M301}$) que possui termos redundantes e contradições lógicas, e deixar o algoritmo aplicar as leis de absorção, idempotência e complemento.

In [2]:
# Definindo as variáveis lógicas como símbolos matemáticos
c1, d1, F, Om = sp.symbols('c1 d1 F Om')

# Expressão Original Não Otimizada:
# (c1 AND ~d1 AND ~F) OR (c1 AND ~d1 AND F AND Om) OR (c1 AND ~c1 AND ~d1)
expr_original = (c1 & ~d1 & ~F) | (c1 & ~d1 & F & Om) | (c1 & ~c1 & ~d1)

print("="*60)
print(" EXPRESSÃO ORIGINAL (Com redundâncias e contradições)")
print("="*60)
print(expr_original)

# Aplicando a otimização lógica nativa do SymPy
expr_otimizada = simplify_logic(expr_original)

print("\n" + "="*60)
print(" EXPRESSÃO OTIMIZADA (Redução de custo computacional)")
print("="*60)
print(expr_otimizada)

 EXPRESSÃO ORIGINAL (Com redundâncias e contradições)
(c1 & ~F & ~d1) | (c1 & ~c1 & ~d1) | (F & Om & c1 & ~d1)

 EXPRESSÃO OTIMIZADA (Redução de custo computacional)
c1 & ~d1 & (Om | ~F)


## 2. Conversão para Formas Normais Canônicas

O CLP pode exigir que a lógica seja estruturada especificamente em **Forma Normal Disjuntiva (FND / SOP)** para lógica em blocos paralelos, ou em **Forma Normal Conjuntiva (FNC / POS)** para matrizes de segurança (*Safety Matrix*). O código abaixo converte a nossa equação otimizada para esses dois padrões.

In [3]:
# Conversão para Forma Normal Disjuntiva (FND / Soma de Produtos)
fnd_expr = to_dnf(expr_otimizada)

# Conversão para Forma Normal Conjuntiva (FNC / Produto de Somas)
fnc_expr = to_cnf(expr_otimizada)

print("="*60)
print(" CONVERSÃO PARA FORMAS CANÔNICAS (FND e FNC)")
print("="*60)

print("\nForma Normal Disjuntiva (FND):")
print("-> Ideal para Diagrama Ladder em paralelo ou blocos OR/AND")
print(fnd_expr)

print("\nForma Normal Conjuntiva (FNC):")
print("-> Ideal para Matriz de Segurança Causa/Efeito (Intersecção)")
print(fnc_expr)

 CONVERSÃO PARA FORMAS CANÔNICAS (FND e FNC)

Forma Normal Disjuntiva (FND):
-> Ideal para Diagrama Ladder em paralelo ou blocos OR/AND
(Om & c1 & ~d1) | (c1 & ~F & ~d1)

Forma Normal Conjuntiva (FNC):
-> Ideal para Matriz de Segurança Causa/Efeito (Intersecção)
c1 & ~d1 & (Om | ~F)
